# TF-IDF で文書ごとの特徴語を取り出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_topics_keywords/tfidf_keywords.ipynb)

NLP クックブック 11-02 ／ トピックとキーワードの抽出

複数の文書があるとき、それぞれの文書を特徴づける単語 (他の文書にはあまり出ない語) を見つけたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, re, urllib.request, zipfile

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

base = "https://www.aozora.gr.jp/cards/"
docs = {
    "吾輩は猫である": aozora(base + "000148/files/789_ruby_5639.zip"),
    "坊っちゃん": aozora(base + "000148/files/752_ruby_2438.zip"),
    "走れメロス": aozora(base + "000035/files/1567_ruby_4948.zip"),
    "銀河鉄道の夜": aozora(base + "000081/files/456_ruby_145.zip"),
}

In [3]:
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import TfidfVectorizer

tagger = Tagger()

def tokenize(text):
    return [w.surface for line in text.split("\n") for w in tagger(line)
            if w.feature.pos1 == "名詞" and w.feature.pos2 in ("普通名詞", "固有名詞") and len(w.surface) > 1]

vec = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, sublinear_tf=True)
X = vec.fit_transform(docs.values())
vocab = vec.get_feature_names_out()

top = {title: [vocab[i] for i in row.toarray().ravel().argsort()[::-1][:10]] for title, row in zip(docs, X)}
pd.DataFrame(top)

,吾輩は猫である,坊っちゃん,走れメロス,銀河鉄道の夜
0,迷亭,山嵐,メロス,ジョバンニ
1,寒月,うらなり,セリヌンティウス,カムパネルラ
2,細君,古賀,濁流,いっぱい
3,主人,宿直,王城,銀河
4,金田,校長,出発,鳥捕り
5,東風,バッタ,信実,あかり
6,沙弥,送別,いま,ほんとう
7,鈴木,堀田,老爺,いろ
8,独仙,シャツ,祝宴,苹果
9,逆上,坊っ,花婿,いま


- 自前のトークナイザを渡すときは `token_pattern=None` を付ける (警告を消す)。
- `sublinear_tf=True` は頻度を 1+log(tf) に圧縮する。長い文書で頻出語が支配的になるのを防ぐ。
- 文書数が少ないと IDF があまり効かない。段落や章を 1 文書とみなすと粒度を変えられる。